# Stratified Sample Fetcher

Loads the GovData CKAN snapshot, filters NetCDF datasets, classifies geo vs. non-geo, draws a stratified random sample, and saves each dataset as RDF/XML.

In [1]:
# ============================================================================
# CONFIG — adjust before running
# ============================================================================
SNAPSHOT_FILE       = "data/raw/govdata_ckan_snapshot2026-05-15_09-57-31.jsonl"
RANDOM_SEED         = 67
SAMPLE_SIZE         = 50   # total datasets to fetch
SAMPLES_PER_SECONDARY = 5  # per secondary stratum (top-4 publishers + rest)
# ============================================================================

In [2]:
import json
from pathlib import Path

import pandas as pd

snapshot_path = Path(SNAPSHOT_FILE)
if not snapshot_path.exists():
    raise FileNotFoundError(f"{snapshot_path} not found.")

records = [
    json.loads(line)
    for line in snapshot_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
df = pd.json_normalize(records)
print(f"Loaded:              {len(df):,} datasets")


def _has_netcdf(row) -> bool:
    for res in (row.get("resources") or []):
        if not isinstance(res, dict):
            continue
        fmt  = (res.get("format")   or "").lower()
        url  = (res.get("url") or res.get("download_url") or res.get("access_url") or "").lower()
        mime = (res.get("mimetype") or "").lower()
        if "netcdf" in fmt or ".nc" in fmt:
            return True
        if url.endswith(".nc"):
            return True
        if "netcdf" in mime:
            return True
    return False


df = df[~df.apply(_has_netcdf, axis=1)].reset_index(drop=True)
print(f"After NetCDF filter: {len(df):,} datasets")

Loaded:              151,579 datasets
After NetCDF filter: 149,786 datasets


In [3]:
import re

GEO_FORMAT_PATTERNS = {
    "WMS":        [r"\bWMS\b", r"WMS_SRVC", r"OGC/WMS", r"application/vnd\.ogc\.wms_xml", r"SERVICE=WMS"],
    "WFS":        [r"\bWFS\b", r"WFS_SRVC", r"OGC/WFS", r"application/vnd\.ogc\.wfs_xml", r"SERVICE=WFS"],
    "WCS":        [r"\bWCS\b", r"WCS_SRVC", r"SERVICE=WCS"],
    "WMTS":       [r"\bWMTS\b", r"WMTS_SRVC", r"application/vnd\.ogc\.wmts_xml", r"SERVICE=WMTS"],
    "GML":        [r"\bGML\b", r"application/gml\+xml", r"application/vnd\.ogc\.gml", r"GML Application Schema", r"INSPIRE-GML", r"XML/GML", r"gml\+xml"],
    "GeoJSON":    [r"\bGEOJSON\b", r"\bGeoJSON\b", r"application/vnd\.geo\+json", r"application/geo\+json", r"geo\+json"],
    "Shapefile":  [r"\bSHP\b", r"\bSHAPE\b", r"\bShape\b", r"Shapefile", r"Shape File", r"ESRI[- ]?Shape", r"gezippte shape", r"gezippte shapedatei", r"\bshp\b"],
    "GeoPackage": [r"\bGPKG\b", r"GeoPackage", r"Geopackage", r"geopackage", r"application/geopackage\+sqlite3", r"application/gpkg"],
    "KML/KMZ":    [r"\bKML\b", r"\bKMZ\b"],
    "GeoTIFF":    [r"\bGEOTIFF\b", r"GeoTIFF", r"GeoTIF", r"Geo-TIFF", r"Cloud Optimized GeoTIFF", r"\bCOG\b", r"georef\. TIFF"],
    "DXF/DWG":    [r"\bDXF\b", r"\bDWG\b", r"dwg/dxf"],
    "CityGML":    [r"CityGML", r"CITYGML", r"citygml"],
    "NAS":        [r"\bNAS\b", r"\bnas\b"],
    "GPX":        [r"\bGPX\b", r"\bgpx\b"],
    "WKT":        [r"\bWKT\b"],
    "LAS/LAZ":    [r"\bLAS\b", r"\bLAZ\b", r"LAS/LAZ"],
    "MapInfo":    [r"MapInfo", r"\bMIF\b", r"\bTAB\b"],
    "ArcGIS/ESRI":[r"ESRI Hub", r"esri hub", r"ArcGIS", r"arcgis", r"File ?Geodatabase", r"Geodatabase", r"GDB", r"MDB", r"SDE-Feature-Class"],
    "GeoRaster":  [r"GRID_ASCII", r"ASCII-Grid", r"XYZ-ASCII", r"\bGRIB\b", r"\bBUFR\b"],
}


def _classify_geo(fmt: str):
    fmt = (fmt or "").strip()
    if not fmt:
        return None
    for group, patterns in GEO_FORMAT_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, fmt, re.IGNORECASE):
                return group
    return None


geo_rows = []
for idx, row in df.iterrows():
    for res in (row.get("resources") or []):
        if not isinstance(res, dict):
            continue
        geo_group = _classify_geo(str(res.get("format") or ""))
        if geo_group:
            geo_rows.append({"dataset_index": idx, "geo_format_group": geo_group})
            break  # one match per dataset is enough to mark it as geo

geo_distribution_rows = pd.DataFrame(geo_rows)
geo_dataset_indices = (
    set(geo_distribution_rows["dataset_index"].unique())
    if not geo_distribution_rows.empty
    else set()
)
print(f"Geo datasets:     {len(geo_dataset_indices):,}")
print(f"Non-geo datasets: {len(df) - len(geo_dataset_indices):,}")

Geo datasets:     25,416
Non-geo datasets: 124,370


In [ ]:
from datetime import datetime
import hashlib
import re

import requests


def _create_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": "govdata-stratified-sampler/0.1",
                       "Accept": "application/rdf+xml, application/xml;q=0.9, */*;q=0.1"})
    return s


def _sanitize(value: str) -> str:
    value = (value or "").strip()
    value = re.sub(r"[^A-Za-z0-9._-]+", "-", value)
    return value.strip("-") or "dataset"


def _fetch_rdf_xml(session: requests.Session, name: str, timeout: int = 60) -> bytes:
    for pattern in [
        "https://www.govdata.de/dataset/{n}.rdf",
        "https://www.govdata.de/ckan/dataset/{n}.rdf",
        "https://govdata.de/dataset/{n}.rdf",
        "https://govdata.de/ckan/dataset/{n}.rdf",
        "https://www.govdata.de/dataset/{n}.xml",
        "https://www.govdata.de/ckan/dataset/{n}.xml",
        "https://govdata.de/dataset/{n}.xml",
        "https://govdata.de/ckan/dataset/{n}.xml",
    ]:
        url = pattern.format(n=name)
        try:
            r = session.get(url, timeout=timeout, allow_redirects=True)
        except requests.RequestException:
            continue
        if r.status_code != 200:
            continue
        ct   = (r.headers.get("Content-Type") or "").lower()
        body = r.content[:1000].lower()
        if b"<rdf:rdf" in body or "application/rdf+xml" in ct or "xml" in ct:
            return r.content
    raise RuntimeError(f"No RDF/XML endpoint reachable for '{name}'.")


def _publisher(row: pd.Series) -> str:
    for src in [
        row.get("organization.name"),
        (row.get("organization") or {}).get("name") if isinstance(row.get("organization"), dict) else None,
        (row.get("organization") or {}).get("title") if isinstance(row.get("organization"), dict) else None,
        row.get("maintainer"),
        row.get("author"),
    ]:
        if isinstance(src, str) and src.strip():
            return src.strip()
    return ""


# --- Build sampling frame ---
geo_indices = set(geo_dataset_indices)
frame_rows = []
for idx, row in df.iterrows():
    name = row.get("name")
    if not isinstance(name, str) or not name.strip():
        continue
    pub = _publisher(row)
    if not pub:
        continue
    frame_rows.append({
        "dataset_index": idx,
        "dataset_id":    row.get("id"),
        "dataset_name":  name.strip(),
        "publisher":     pub,
        "primary_stratum": "geo" if idx in geo_indices else "non_geo",
    })

sample_df = pd.DataFrame(frame_rows)
if sample_df.empty:
    raise RuntimeError("No candidates with name+publisher found.")

print(f"Candidates: {len(sample_df):,}")
print(sample_df["primary_stratum"].value_counts().to_string())
print()

# --- Assign secondary strata (top-4 publishers per primary) ---
PRIMARY_STRATA = ["geo", "non_geo"]
top4_map = {}
for primary in PRIMARY_STRATA:
    sub = sample_df[sample_df["primary_stratum"] == primary]
    top4_map[primary] = sub["publisher"].value_counts().head(4).index.tolist()

sample_df["secondary_stratum"] = sample_df.apply(
    lambda r: r["publisher"] if r["publisher"] in top4_map[r["primary_stratum"]] else "rest",
    axis=1,
)

# --- Draw reproducible sample ---
selected_parts = []
for primary in PRIMARY_STRATA:
    sub = sample_df[sample_df["primary_stratum"] == primary].copy()
    print(f"Sampling {primary.upper()}:")
    for sec in top4_map[primary] + ["rest"]:
        stratum = sub[sub["secondary_stratum"] == sec]
        n = min(SAMPLES_PER_SECONDARY, len(stratum))
        if n == 0:
            continue
        key = f"{RANDOM_SEED}-{primary}-{sec}"
        seed = int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**31 - 1)
        selected_parts.append(stratum.sample(n=n, random_state=seed))
        print(f"  {sec:30s} -> {n}")

selected_df = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()
if selected_df.empty:
    raise RuntimeError("Sampling produced no results.")

# Fill up to SAMPLE_SIZE if needed
if len(selected_df) < SAMPLE_SIZE:
    pool = sample_df[~sample_df["dataset_index"].isin(set(selected_df["dataset_index"]))]
    if not pool.empty:
        fill = pool.sample(n=min(SAMPLE_SIZE - len(selected_df), len(pool)), random_state=RANDOM_SEED).copy()
        fill["secondary_stratum"] = fill["secondary_stratum"] + "_fill"
        selected_df = pd.concat([selected_df, fill], ignore_index=True)

if len(selected_df) > SAMPLE_SIZE:
    selected_df = selected_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)

print()
print(f"Final sample: {len(selected_df)} datasets (seed={RANDOM_SEED})")
print(selected_df.groupby(["primary_stratum", "secondary_stratum"]).size().to_string())
print()

# --- Fetch and save RDF/XML ---
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
out_dir = Path("../../data") / f"sample_{timestamp}"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving to: {out_dir}")
print()

session = _create_session()
saved, failed = [], []
counter = {}

for i, (_, row) in enumerate(selected_df.iterrows(), 1):
    primary = row["primary_stratum"]
    sec_raw = str(row["secondary_stratum"])
    sec     = _sanitize(sec_raw.lower())
    name    = row["dataset_name"]

    key = (primary, sec)
    counter[key] = counter.get(key, 0) + 1
    seq = counter[key]
    fname = f"{primary}_{sec}.rdf" if seq == 1 else f"{primary}_{sec}_{seq:02d}.rdf"

    try:
        rdf_xml = _fetch_rdf_xml(session, name)
        (out_dir / fname).write_bytes(rdf_xml)
        saved.append({"dataset_index": row["dataset_index"], "dataset_id": row["dataset_id"],
                       "dataset_name": name, "publisher": row["publisher"],
                       "primary_stratum": primary, "secondary_stratum": sec_raw,
                       "file_path": str(out_dir / fname)})
        print(f"[{i:2d}/{len(selected_df)}] OK   {name}")
    except Exception as exc:
        failed.append({"dataset_index": row["dataset_index"], "dataset_id": row["dataset_id"],
                        "dataset_name": name, "error": str(exc)})
        print(f"[{i:2d}/{len(selected_df)}] FAIL {name} ({exc})")

saved_df  = pd.DataFrame(saved)
failed_df = pd.DataFrame(failed)

manifest = out_dir / "sample_manifest.csv"
saved_df.to_csv(manifest, index=False)

print()
print("=" * 70)
print(f"Saved: {len(saved_df)} RDF files | Failed: {len(failed_df)}")
print(f"Manifest: {manifest}")
print(f"Output:   {out_dir}")
print("=" * 70)

Candidates: 149,786
primary_stratum
non_geo    124370
geo         25416

Sampling GEO:
  open-data-portal-rheinland-pfalz -> 5
  transparenzportal-hamburg      -> 5
  gdi-de                         -> 5
  open-data-brandenburg          -> 5
  rest                           -> 5
Sampling NON_GEO:
  land-schleswig-holstein        -> 5
  open-data-bayern               -> 5
  gdi-de                         -> 5
  open-data-brandenburg          -> 5
  rest                           -> 5

Final sample: 50 datasets (seed=67)
primary_stratum  secondary_stratum               
geo              gdi-de                              5
                 open-data-brandenburg               5
                 open-data-portal-rheinland-pfalz    5
                 rest                                5
                 transparenzportal-hamburg           5
non_geo          gdi-de                              5
                 land-schleswig-holstein             5
                 open-data-bayern        